# Audit Langkah 1b — Kaggle *(REVISI_12NOV; hanya membaca)*

Sel di bawah hanya membaca informasi. Tidak ada yang dipasang atau ditulis. Yang dicetak:
- `nvidia-smi`;
- versi Python dan *package* bawaan Kaggle;
- GPU yang terlihat oleh torch;
- revisi *checkpoint* IndoBERT di Hub.

**Langkah:**
1. Di panel kanan *Settings*, pilih *Accelerator* **GPU T4 x2** dan nyalakan **Internet**. Internet memerlukan verifikasi nomor HP. Tidak perlu menambahkan dataset.
2. Jalankan sel di bawah, lalu salin seluruh output-nya.
3. Matikan sesi (*Stop session*) supaya kuota GPU tidak terpakai.

In [ ]:
# ══ Tahap 10–11 — Audit Langkah 1b REVISI_12NOV: Kaggle (GPU T4 x2, Internet On; hanya membaca, tidak memasang apa pun) ══
# Package: os, re, sys, json, shutil, platform, subprocess, urllib, importlib.metadata (bawaan Python); torch (bawaan Kaggle)
import os, re, sys, json, shutil, platform, subprocess, urllib.request, importlib.metadata as im
import torch
CK, CUDA_COLAB = "indobenchmark/indobert-base-p1", (13, 0)   # checkpoint CFG; torch Colab 2.14.1+cu130 (Data: Tahap 0.1)
PK = ["torch", "transformers", "tokenizers", "huggingface-hub", "numpy", "pandas", "scikit-learn", "scipy", "pyarrow",
      "openpyxl", "matplotlib"]


def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return "tidak terpasang"


# 1) nvidia-smi (driver dan versi CUDA tertinggi yang didukung driver) dan GPU yang terlihat oleh torch bawaan
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
cuda_drv = (re.findall(r"CUDA Version:\s*([\d.]+)", smi) or ["0.0"])[0]
gpu = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]

# 2) Revisi checkpoint di Hub dilihat dari Kaggle (sekaligus cek Internet On)
try:
    with urllib.request.urlopen(f"https://huggingface.co/api/models/{CK}", timeout=30) as r:
        info = json.load(r)
    hub, hub_t = info.get("sha"), str(info.get("lastModified", "-"))[:19]
except Exception as e:                                      # Internet Off atau Hub tidak terjangkau: dilaporkan
    hub, hub_t = None, f"gagal ({type(e).__name__})"

# 3) Ringkasan + cek (lib/bantu belum ada di Kaggle, jadi dicetak langsung)
kaggle = os.path.isdir("/kaggle/input")
print(f"{'═' * 10} AUDIT LANGKAH 1B — KAGGLE (HANYA MEMBACA) {'═' * 10}\n{smi.strip() or 'nvidia-smi tidak ada'}\n{'─' * 48}")
print(f"Platform    : {'kaggle' if kaggle else 'BUKAN Kaggle'} · Python {platform.python_version()} · {sys.executable}")
print(f"CUDA        : driver {cuda_drv} · torch {torch.__version__} (CUDA {torch.version.cuda}) · cuDNN {torch.backends.cudnn.version()} · "
      f"arsitektur torch {' '.join(torch.cuda.get_arch_list()) or '-'}")
print(f"GPU (torch) : {len(gpu)} · {' · '.join(gpu) or 'tidak ada'}")
print(f"Revisi Hub  : {hub or '-'} · commit terakhir {hub_t}\n\n{'Package':<17}Versi bawaan")
print("\n".join(f"{p:<17}{versi(p)}" for p in PK))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("platform", kaggle, "kaggle" if kaggle else "bukan kaggle", "kaggle"),
        ("GPU terlihat · semuanya T4", bool(gpu) and all("T4" in g for g in gpu),
         f"{len(gpu)} · {'ya' if gpu and all('T4' in g for g in gpu) else 'tidak'}", "≥ 1 · ya"),
        ("driver mendukung CUDA build torch Colab", tuple(map(int, cuda_drv.split(".")[:2])) >= CUDA_COLAB,
         cuda_drv, "≥ " + ".".join(map(str, CUDA_COLAB))),
        ("Internet: Hub dapat dihubungi", hub is not None, "ya" if hub else hub_t, "ya")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")